In [ ]:
import pandas as pd
import networkx as nx
import json
from datetime import datetime

# 행동 데이터와 뉴스 데이터 로드
behavior_data = pd.read_csv(
    r"C:\Users\shw\Desktop\스트리밍세미나\behaviors_sortedByTime.tsv",
    sep='\t', header=None
)
news_data = pd.read_csv(
    r"C:\Users\shw\Desktop\스트리밍세미나\news.tsv",
    sep='\t', header=None
)

# 컬럼 이름 지정
behavior_data.columns = ['Impression ID', 'User ID', 'Time', 'History', 'Impressions']
news_data.columns = ['News ID', 'Category', 'SubCategory', 'Title', 'Abstract', 'URL', 
                     'Title Entities', 'Abstract Entities']

# 데이터 미리보기
print("Behavior Data Sample:")
print(behavior_data.head())
print("\nNews Data Sample:")
print(news_data.head())


SyntaxError: (unicode error) 'unicodeescape' codec can't decode bytes in position 2-3: truncated \UXXXXXXXX escape (2344342838.py, line 8)

In [ ]:
# 3일치 데이터 필터링: 2019-11-09 ~ 2019-11-12
start_date = datetime(2019, 11, 9)
end_date = datetime(2019, 11, 12)
behavior_data['Time'] = pd.to_datetime(behavior_data['Time'])
behavior_data = behavior_data[(behavior_data['Time'] >= start_date) & (behavior_data['Time'] <= end_date)]
print("총 {}개의 데이터를 가져왔습니다.".format(behavior_data.shape[0]))


총 61417개의 데이터를 가져왔습니다.


In [ ]:
# 그래프 생성
G = nx.Graph()

# 뉴스 노드 추가
for _, row in news_data.iterrows():
    news_id = row['News ID']
    news_features = {
        'type': 'news',
        'category': row['Category'],
        'subcategory': row['SubCategory'],
        'title': row['Title'],
        'abstract': row['Abstract']
    }
    
    # --- 타이틀 엔티티 처리 ---
    try:
        title_entities = json.loads(row['Title Entities'])
    except Exception:
        title_entities = []
    
    title_entity_labels = []
    title_entity_types = []
    title_entity_surfaceforms = []
    for entity in title_entities:
        label = entity.get('Label')
        entity_type = entity.get('Type')
        surface_forms = entity.get('SurfaceForms')
        if label is not None:
            title_entity_labels.append(label)
        if entity_type is not None:
            title_entity_types.append(entity_type)
        if surface_forms is not None:
            # surface_forms가 리스트가 아닐 수도 있으므로 리스트화
            if isinstance(surface_forms, list):
                title_entity_surfaceforms.extend(surface_forms)
            else:
                title_entity_surfaceforms.append(surface_forms)
    
    news_features['title_entity_labels'] = title_entity_labels
    news_features['title_entity_types'] = title_entity_types
    news_features['title_entity_surfaceforms'] = title_entity_surfaceforms
    
    # --- 앱스트랙트 엔티티 처리 ---
    try:
        abstract_entities = json.loads(row['Abstract Entities'])
    except Exception:
        abstract_entities = []
    
    abstract_entity_labels = []
    abstract_entity_types = []
    abstract_entity_surfaceforms = []
    for entity in abstract_entities:
        label = entity.get('Label')
        entity_type = entity.get('Type')
        surface_forms = entity.get('SurfaceForms')
        if label is not None:
            abstract_entity_labels.append(label)
        if entity_type is not None:
            abstract_entity_types.append(entity_type)
        if surface_forms is not None:
            if isinstance(surface_forms, list):
                abstract_entity_surfaceforms.extend(surface_forms)
            else:
                abstract_entity_surfaceforms.append(surface_forms)
    
    news_features['abstract_entity_labels'] = abstract_entity_labels
    news_features['abstract_entity_types'] = abstract_entity_types
    news_features['abstract_entity_surfaceforms'] = abstract_entity_surfaceforms
    
    G.add_node(news_id, **news_features)


In [ ]:
# 유저별 피처(파생 피처) 집계를 위한 사전 초기화
user_features = {}

for _, row in behavior_data.iterrows():
    user_id = row['User ID']
    
    # 유저가 처음 등장하면 초기화
    if user_id not in user_features:
        user_features[user_id] = {
            'history_count': 0,         # History에 나온 뉴스 개수 총합
            'impressions_total': 0,     # Impressions에 나온 전체 뉴스 개수
            'impressions_clicked': 0,   # 클릭된(=1) Impressions 개수
            'active_times': []          # 해당 유저의 모든 활동 시간 기록
        }
    
    # 활동 시간 기록
    user_features[user_id]['active_times'].append(row['Time'])
    
    # History 처리: 뉴스 ID들이 공백으로 구분되어 있음
    history = row['History']
    if pd.notna(history) and history.strip() != "":
        history_items = history.split()
        user_features[user_id]['history_count'] += len(history_items)
    
    # Impressions 처리: 각 항목은 "뉴스ID-클릭여부" 형태
    impressions = row['Impressions']
    if pd.notna(impressions) and impressions.strip() != "":
        impressions_items = impressions.split()
        user_features[user_id]['impressions_total'] += len(impressions_items)
        for imp in impressions_items:
            parts = imp.rsplit('-', 1)
            if len(parts) == 2:
                try:
                    clicked = int(parts[1])
                except:
                    clicked = None
                if clicked == 1:
                    user_features[user_id]['impressions_clicked'] += 1


In [ ]:
for user_id, feats in user_features.items():
    # 마지막 활동 시간을 기준으로 파생 피처 생성
    last_active_time = max(feats['active_times']) if feats['active_times'] else None
    hour_of_day = last_active_time.hour if last_active_time else None
    day_of_week = last_active_time.strftime('%A') if last_active_time else None
    impressions_total = feats['impressions_total']
    impressions_clicked = feats['impressions_clicked']
    ctr = impressions_clicked / impressions_total if impressions_total > 0 else 0.0
    
    G.add_node(user_id, type='user',
               history_count=feats['history_count'],
               impressions_total=impressions_total,
               impressions_clicked=impressions_clicked,
               ctr=ctr,
               last_active_time=last_active_time,
               hour_of_day=hour_of_day,
               day_of_week=day_of_week)


In [ ]:
for _, row in behavior_data.iterrows():
    user_id = row['User ID']
    union_news = set()
    
    # History에서 뉴스 ID 추출
    history = row['History']
    if pd.notna(history) and history.strip() != "":
        union_news |= set(history.split())
    
    # Impressions에서 클릭된(=1) 뉴스 ID 추출
    impressions = row['Impressions']
    if pd.notna(impressions) and impressions.strip() != "":
        for imp in impressions.split():
            parts = imp.rsplit('-', 1)
            if len(parts) == 2:
                news_id, clicked_str = parts
                try:
                    clicked = int(clicked_str)
                except:
                    clicked = None
                if clicked == 1:
                    union_news.add(news_id)
    
    # 엣지 추가 (뉴스 노드가 존재할 때만)
    for news_id in union_news:
        if G.has_node(news_id):
            G.add_edge(user_id, news_id)


In [ ]:
# 유저 노드와 뉴스 노드 개수 확인
user_nodes = [n for n, attr in G.nodes(data=True) if attr.get('type') == 'user']
news_nodes = [n for n, attr in G.nodes(data=True) if attr.get('type') == 'news']

print("유저 노드 개수:", len(user_nodes))
print("뉴스 노드 개수:", len(news_nodes))
print("엣지 개수:", G.number_of_edges())

# (선택) 샘플로 유저와 뉴스 노드 3개씩 출력하여 피처 확인
print("\nSample User Nodes:")
for node in user_nodes[:3]:
    print("User Node ID:", node)
    print("Features:", G.nodes[node])
    print("-" * 50)

print("\nSample News Nodes:")
for node in news_nodes[:3]:
    print("News Node ID:", node)
    print("Features:", G.nodes[node])
    print("-" * 50)


유저 노드 개수: 30571
뉴스 노드 개수: 51282
엣지 개수: 726950

Sample User Nodes:
User Node ID: U65916
Features: {'type': 'user', 'history_count': 30, 'impressions_total': 86, 'impressions_clicked': 4, 'ctr': 0.046511627906976744, 'last_active_time': Timestamp('2019-11-10 23:05:47'), 'hour_of_day': 23, 'day_of_week': 'Sunday'}
--------------------------------------------------
User Node ID: U49985
Features: {'type': 'user', 'history_count': 558, 'impressions_total': 353, 'impressions_clicked': 23, 'ctr': 0.06515580736543909, 'last_active_time': Timestamp('2019-11-11 20:13:52'), 'hour_of_day': 20, 'day_of_week': 'Monday'}
--------------------------------------------------
User Node ID: U25550
Features: {'type': 'user', 'history_count': 7, 'impressions_total': 5, 'impressions_clicked': 1, 'ctr': 0.2, 'last_active_time': Timestamp('2019-11-09 00:02:44'), 'hour_of_day': 0, 'day_of_week': 'Saturday'}
--------------------------------------------------

Sample News Nodes:
News Node ID: N55528
Features: {'typ

In [ ]:
def update_user_features(user_id, new_row, user_features):
    """
    새로운 행동(new_row)이 들어왔을 때, 기존 user_features를 업데이트하는 함수.
    """
    # 유저가 존재하지 않으면 초기화
    if user_id not in user_features:
        user_features[user_id] = {
            'history_count': 0,
            'impressions_total': 0,
            'impressions_clicked': 0,
            'active_times': []
        }
    feats = user_features[user_id]
    
    # 활동 시간 추가
    feats['active_times'].append(new_row['Time'])
    
    # History 업데이트
    history = new_row['History']
    if pd.notna(history) and history.strip() != "":
        feats['history_count'] += len(history.split())
    
    # Impressions 업데이트
    impressions = new_row['Impressions']
    if pd.notna(impressions) and impressions.strip() != "":
        items = impressions.split()
        feats['impressions_total'] += len(items)
        for imp in items:
            parts = imp.rsplit('-', 1)
            if len(parts) == 2:
                try:
                    clicked = int(parts[1])
                except:
                    clicked = None
                if clicked == 1:
                    feats['impressions_clicked'] += 1

# 사용 예시:
# new_row = behavior_data.iloc[0]  # 새로운 행 (예시)
# update_user_features(new_row['User ID'], new_row, user_features)


In [ ]:
from transformers import BertTokenizer, BertModel
import torch
import pandas as pd
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
bert_model = BertModel.from_pretrained("bert-base-uncased")
bert_model.to(device)
bert_model.eval()

def get_text_embedding(text):
    if not text or pd.isna(text):
        return torch.zeros(768, device=device).cpu()
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=128)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = bert_model(**inputs)
    embedding = outputs.last_hidden_state[:, 0, :]
    return embedding.squeeze(0).cpu()

def get_batch_embeddings(texts, batch_size=32, device=device):
    embeddings = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Batch Embedding"):
        batch_texts = texts[i:i+batch_size]
        inputs = tokenizer(batch_texts, return_tensors="pt", truncation=True, padding=True, max_length=128)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = bert_model(**inputs)
        batch_emb = outputs.last_hidden_state[:, 0, :]
        embeddings.append(batch_emb.cpu())
    return torch.cat(embeddings, dim=0)


In [ ]:
news_fields = [
    "category",
    "subcategory",
    "title",
    "abstract",
    "title_entity_labels",
    "title_entity_types",
    "title_entity_surfaceforms",
    "abstract_entity_labels",
    "abstract_entity_types",
    "abstract_entity_surfaceforms"
]

news_embedding_cache = {}

def precompute_news_embeddings(G, fields, batch_size=32):
    news_nodes = [node for node, attr in G.nodes(data=True) if attr.get("type") == "news"]
    field_embeddings = {}
    for field in tqdm(fields, desc="Processing fields"):
        texts = []
        for node in news_nodes:
            attr = G.nodes[node]
            val = attr.get(field, "")
            if isinstance(val, list):
                text_val = " ".join(map(str, val))
            else:
                text_val = str(val)
            texts.append(text_val)
        emb = get_batch_embeddings(texts, batch_size=batch_size, device=device)
        field_embeddings[field] = emb
    news_embeddings = {}
    for i, node in enumerate(tqdm(news_nodes, desc="Concatenating embeddings per news node")):
        concatenated = torch.cat([field_embeddings[field][i] for field in fields], dim=0)
        news_embeddings[node] = concatenated
    return news_embeddings

news_precomputed = precompute_news_embeddings(G, news_fields, batch_size=32)


Processing fields:   0%|          | 0/10 [01:09<?, ?it/s]


KeyboardInterrupt: 

In [ ]:
def build_user_feature(node_attrs):
    numeric_fields = ["history_count", "impressions_total", "impressions_clicked", "ctr", "hour_of_day"]
    numeric_vals = [float(node_attrs.get(f, 0)) for f in numeric_fields]
    numeric_tensor = torch.tensor(numeric_vals, dtype=torch.float)
    
    day_to_int = {"Monday": 1, "Tuesday": 2, "Wednesday": 3,
                  "Thursday": 4, "Friday": 5, "Saturday": 6, "Sunday": 7}
    day_str = node_attrs.get("day_of_week", "")
    day_numeric = float(day_to_int.get(day_str, 0))
    day_tensor = torch.tensor([day_numeric], dtype=torch.float)
    
    last_active = node_attrs.get("last_active_time", "")
    last_active_text = str(last_active) if last_active else ""
    last_active_emb = get_text_embedding(last_active_text)
    
    return torch.cat([numeric_tensor, day_tensor, last_active_emb], dim=0)

def build_news_feature(node_attrs):
    node_id = node_attrs.get("News ID")
    if node_id in news_precomputed:
        return news_precomputed[node_id]
    else:
        embeddings = []
        for field in news_fields:
            val = node_attrs.get(field, "")
            if isinstance(val, list):
                text_val = " ".join(map(str, val))
            else:
                text_val = str(val)
            emb = get_text_embedding(text_val)
            embeddings.append(emb)
        return torch.cat(embeddings, dim=0)

def build_node_feature(node_attrs):
    if node_attrs.get("type") == "user":
        return build_user_feature(node_attrs)
    elif node_attrs.get("type") == "news":
        return build_news_feature(node_attrs)
    else:
        return torch.tensor([0.0], dtype=torch.float)


In [ ]:
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm

def process_node(node_attr_tuple):
    """
    각 노드와 그 속성을 받아 build_node_feature 함수를 적용한 후,
    (node, feature_vector)를 반환하는 함수.
    """
    node, attr = node_attr_tuple
    return node, build_node_feature(attr)

# 모든 노드 데이터를 리스트로 변환
nodes_data = list(G.nodes(data=True))
num_nodes = len(nodes_data)

# ThreadPoolExecutor를 사용하여 병렬 처리 (예: max_workers=8)
with ThreadPoolExecutor(max_workers=8) as executor:
    # executor.map은 순서대로 결과를 반환합니다.
    results = list(tqdm(executor.map(process_node, nodes_data),
                        total=num_nodes,
                        desc="Building node features"))

# 결과를 그래프에 다시 반영
for node, feature in results:
    G.nodes[node]['x'] = feature


In [ ]:
from torch_geometric.utils import from_networkx

data = from_networkx(G)
print("PyTorch Geometric Data 객체:")
print(data)
# data.x: 노드 피쳐 행렬, data.edge_index: 엣지 정보


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch_geometric.utils import from_networkx
from tqdm import tqdm

# GCN 모델 정의
class SimpleGCN(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super(SimpleGCN, self).__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, out_channels)
    
    def forward(self, x, edge_index):
        # 첫 번째 GCN 레이어: 이웃 노드의 피처 집계 후 ReLU 활성화
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        # 두 번째 GCN 레이어: 추가 집계
        x = self.conv2(x, edge_index)
        return x

# Data 객체 (data.x: 노드 피처 행렬, data.edge_index: 엣지 정보)가 준비되어 있다고 가정
# 예시로, 더미 레이블을 생성합니다.
num_nodes = data.x.size(0)
data.y = torch.zeros(num_nodes, dtype=torch.long)

# 사용할 디바이스 결정 및 Data 객체를 해당 디바이스로 이동
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
data = data.to(device)

# 모델 초기화
in_channels = data.x.size(1)
hidden_channels = 128
out_channels = 3  # 예시: 3개의 클래스로 분류
model = SimpleGCN(in_channels, hidden_channels, out_channels)

# 여러 GPU 사용: DataParallel 적용
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs.")
    model = nn.DataParallel(model)
else:
    print("Using single GPU or CPU.")

model = model.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# 학습 루프 (예시로 100 에포크, tqdm으로 진행 표시)
num_epochs = 100
model.train()
for epoch in tqdm(range(num_epochs), desc="Training epochs"):
    optimizer.zero_grad()
    out = model(data.x, data.edge_index)
    loss = F.cross_entropy(out, data.y)
    loss.backward()
    optimizer.step()
    
    # 10 에포크마다 손실 출력 (tqdm.write 사용)
    if epoch % 10 == 0:
        tqdm.write(f"Epoch {epoch}, Loss: {loss.item()}")

# 학습 후 평가 (출력 확인)
model.eval()
with torch.no_grad():
    final_out = model(data.x, data.edge_index)
print("Final model output (first 10 nodes):")
print(final_out[:10])
